# 09 — Target load and gateway-capacity envelope

E-Perf-1 is the matched 1,000 msg/s target-load comparison, reported as the Target-Load Delivery Summary. E-Perf-10 is the common-grid gateway envelope. Offered rate, achieved rate, pooled loss, p99 latency, delivery ceiling, normalized p99 knee, and MQTT support-path censoring remain separate outputs. No plot is described as maximum sustainable SUT throughput beyond the support path. Explicit diagnostic inputs are thesis_evidence=false with descriptive-only uncertainty; missing inputs render PENDING, never zero.


In [ ]:
import json, os, re
import matplotlib.pyplot as plt
import pandas as pd
from wafer_analysis.canonical import capacity_tables, target_latency_table
from wafer_analysis.focused import evidence_label, pending_record, passed_artifacts, percentile_rows, target_load_rows
from wafer_analysis.paths import find_canonical_ledger, resolve_analysis_batch
from wafer_analysis.plots import SYSTEM_COLORS, save_figure
from wafer_analysis.tables import save_table

target_batch,target_canonical=resolve_analysis_batch('e-perf-1',os.environ.get('E_PERF_1_DIR'))
target_raw=pd.DataFrame() if target_batch is None else (target_load_rows(target_batch) if target_canonical else percentile_rows(target_batch))
if target_canonical:
    target_records=[]
    for row in target_raw.to_dict('records'):
        match=re.search(r'run-(\d+)',str(row['run']))
        if match is None: raise ValueError(f"malformed canonical run name: {row['run']}")
        target_records.append({**row,'run_index':int(match.group(1))})
    target=target_latency_table(target_records)
elif target_raw.empty: target=pd.DataFrame([pending_record('matched 1,000 msg/s target load','no passed E-Perf-1 leaf','nanoseconds')])
else: target=(target_raw.groupby('condition').agg(N_runs=('run','nunique'),median_p95_ns=('p95_ns','median')).reset_index()); target['units']='nanoseconds'; target['estimator']='diagnostic median run p95'; target['uncertainty']='descriptive only'; target['claim_boundary']='matched 1,000 msg/s target load; not capacity'; target['thesis_evidence']=False
display(target)

batch,canonical=resolve_analysis_batch('e-perf-10',os.environ.get('E_PERF_10_DIR'))
summary=None
if batch is not None:
    summaries=passed_artifacts(batch,'rate-sweep-summary.json')
    if summaries: summary=summaries[0][1]
if canonical:
    summary_path=find_canonical_ledger(os.environ['WAFER_EVAL_BATCH_ID'])/'rate-sweep-summary.json'
    summary=json.loads(summary_path.read_text())
if summary is None:
    display(pd.DataFrame([pending_record('gateway-capacity envelope','final or diagnostic summary is unavailable','messages/second, fraction, nanoseconds')]))
else:
    if summary.get('thesis_evidence') is True:
        rates,boundaries=capacity_tables(summary)
        if not canonical:
            rates['thesis_evidence']=False; rates['estimator']='diagnostic preview of final schema'; boundaries['thesis_evidence']=False; boundaries['claim_boundary']='diagnostic preview; not final capacity evidence'
    else:
        raw=[] if batch is None else [value for _,value in passed_artifacts(batch,'rate-sweep.json')]
        raw=pd.DataFrame([{'system':value['system'],'offered_rate_msg_s':value['offered_rate_msg_s'],'achieved_rate_msg_s':value['achieved_rate_msg_s'],'loss_percent':value['loss_percent'],'p99_ns':value['latency_ns']['p99']} for value in raw])
        rates=(raw.groupby(['system','offered_rate_msg_s']).agg(N_runs=('system','size'),median_achieved_rate_msg_s=('achieved_rate_msg_s','median'),pooled_loss=('loss_percent','median'),median_p99_ns=('p99_ns','median')).reset_index()) if not raw.empty else pd.DataFrame()
        if not rates.empty: rates['pooled_loss']/=100; rates['units']='messages/second, fraction, nanoseconds'; rates['estimator']='diagnostic descriptive summary'; rates['thesis_evidence']=False
        boundary_rows=[{'system':system,'delivery_ceiling_msg_s':value.get('last_good_rate_msg_s'),'normalized_p99_knee_msg_s':None,'mqtt_support_path_limitation':None,'units':'messages/second','claim_boundary':'diagnostic bounded tested rates; not final capacity','thesis_evidence':False} for system,value in summary.get('systems',{}).items()]
        boundaries=pd.DataFrame(boundary_rows)
    print(evidence_label(int(rates.N_runs.sum()),'messages/second, fraction, nanoseconds',canonical)); display(rates); display(boundaries)
    if not rates.empty:
        fig,axes=plt.subplots(1,3,figsize=(15,4))
        for system,group in rates.groupby('system'):
            group=group.sort_values('offered_rate_msg_s'); color=SYSTEM_COLORS.get({'wafer':'WAFER','native':'Native','ekuiper':'eKuiper'}.get(system,'')); axes[0].plot(group.offered_rate_msg_s,group.median_achieved_rate_msg_s,marker='o',label=system,color=color); axes[1].plot(group.offered_rate_msg_s,group.pooled_loss,marker='o',label=system,color=color); axes[2].plot(group.offered_rate_msg_s,group.median_p99_ns,marker='o',label=system,color=color)
        axes[0].plot([1000,16000],[1000,16000],linestyle='--',color='black',label='offered reference'); axes[0].set_title('Offered versus achieved rate'); axes[0].set_ylabel('Achieved rate (msg/s)')
        axes[1].axhline(.01,linestyle='--',color='black'); axes[1].set_title('Pooled loss'); axes[1].set_ylabel('Loss fraction')
        axes[2].set_yscale('log'); axes[2].set_title('Run-level p99 latency'); axes[2].set_ylabel('p99 (ns, log scale)')
        for ax in axes: ax.set_xlabel('Offered rate (msg/s)')
        axes[0].legend()
        output=os.environ.get('WAFER_ANALYSIS_OUTPUT_DIR')
        if output:
            save_figure(fig,'e-perf-10/gateway-capacity-metrics',output); save_table(rates,'e-perf-10-rate-estimates',output); save_table(boundaries,'e-perf-10-claim-boundaries',output); save_table(target,'e-perf-1-target-load',output)
            if 'median_normalized_p99' in rates:
                knee_fig,knee_ax=plt.subplots(); [knee_ax.plot(group.offered_rate_msg_s,group.median_normalized_p99,marker='o',label=system,color=SYSTEM_COLORS.get({'wafer':'WAFER','native':'Native','ekuiper':'eKuiper'}.get(system,''))) for system,group in rates.groupby('system')]; knee_ax.axhline(2.0,linestyle='--',color='black'); knee_ax.set_xlabel('Offered rate (msg/s)'); knee_ax.set_ylabel('Normalized median run p99'); knee_ax.set_title('Normalized p99 degradation knee'); knee_ax.legend(); save_figure(knee_fig,'e-perf-10/normalized-p99-knee',output)
